# Qwen Past-200 H100 Artifact Pipeline

Run top to bottom on Colab Pro with an H100 runtime. This notebook is now an artifact builder and offline-ranker, not a Mac throughput proof.

Current local context from the 2026-05-26 M3 Pro bench:

| Local finding | Consequence for this notebook |
|---|---|
| Locked Qwen-3B predec stack is the ship path at about 30 dec_tps | Do not replace it with AWQ Option B or W4A8 by default. |
| AWQ Option B plus W4A8 regressed when it disabled predec | Still emit AWQ/Q2/IQ2 calibration artifacts, but mark them experimental. |
| Eagle5 showed no local gain without trace-dispatch evidence | Colab trains/ranks heads offline; acceptance and net TPS must be proven locally. |
| The path past 200 is the 1.5B student plus local Metal work | Spend H100 time on the student corpus, frozen baseline, Eagle5 grid, and quant artifacts. |

What this notebook does on H100:

1. Build/resume Drive-backed corpora with residual, intermediate, top-k logits, and per-site activation stats.
2. Emit AWQ smoothing and Q2/IQ2 importance artifacts as future/experimental inputs.
3. Export frozen HF baselines for Eagle5 training and eval.
4. Train a wider Eagle5 grid and rank it with tau/frontier policy search.
5. Write a compact handoff manifest with local-only gates and commands.

What stays local on the Mac: GGUF conversion, kernel-profile/autotune, trace-dispatch spec validation, clean absolute benches, per-channel LM-head W4A8 calibration, continuous batching, and any ship/no-ship decision.


In [ ]:
# Cell 1 - H100 setup, repo, and run contract
from google.colab import drive
drive.mount('/content/drive')

import glob, gc, json, math, os, shutil, subprocess, sys
from pathlib import Path


def run(cmd, **kwargs):
    cmd = [str(x) for x in cmd]
    print('$ ' + ' '.join(cmd), flush=True)
    return subprocess.run(cmd, check=True, **kwargs)


run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'accelerate>=1.0', 'datasets>=3.0', 'pyarrow>=17', 'tqdm>=4.66',
    'zstandard', 'bitsandbytes>=0.43', 'gguf', 'safetensors>=0.4',
    'hf_transfer>=0.1.9', 'transformers>=4.45',
])

os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

try:
    from google.colab import userdata
    _hf_token = userdata.get('HF_TOKEN')
except Exception:
    _hf_token = None
if _hf_token:
    os.environ['HF_TOKEN'] = _hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = _hf_token
    print('HF_TOKEN loaded from Colab secrets.')
else:
    print('No HF_TOKEN secret found; public Hub downloads still work.')

import numpy as np
import pyarrow
import torch
import transformers
from transformers import AutoConfig

assert torch.cuda.is_available(), 'No CUDA device: choose an H100 GPU runtime.'
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
STRICT_H100 = True
if STRICT_H100 and 'H100' not in GPU_NAME.upper():
    raise RuntimeError(f'This notebook is H100-tuned; current GPU is {GPU_NAME}. Set STRICT_H100=False to override.')
print(f'GPU: {GPU_NAME}  VRAM: {VRAM_GB:.1f} GB')
print(f'transformers={transformers.__version__} torch={torch.__version__} pyarrow={pyarrow.__version__}')

DRIVE_ROOT = '/content/drive/MyDrive/dismantle'
LOCAL_ROOT = '/content/dismantle_work'
REPO_DIR = '/content/dismantle'
os.makedirs(DRIVE_ROOT, exist_ok=True)
drive_free_gb = shutil.disk_usage(DRIVE_ROOT).free / 1e9
content_free_gb = shutil.disk_usage('/content').free / 1e9
print(f'Drive free: {drive_free_gb:.1f} GB; /content free: {content_free_gb:.1f} GB')
if drive_free_gb < 40:
    print('WARN: this run writes large corpora/checkpoints. Reduce target seqs or run one target at a time if Drive fills.')

REPO = 'https://github.com/joshuahickscorp/dismantle.git'
BRANCH = 'main'  # Set to your pushed workspace branch/commit before running.

# Run contract. The default H100 spend is the 1.5B student path. Qwen-3B is
# disabled by default because the latest local bench already identified the
# shipping Qwen-3B config: locked predec fast path, no AWQ/spec promotion yet.
RUN_STUDENT_SINGLE_SESSION = True
RUN_QWEN3B_REFERENCE = False  # Optional offline diagnostic only; not a ship projection.
BUILD_CORPUS_IF_NEEDED = True
RUN_QUANT_ARTIFACTS = True
RUN_EAGLE5_GRID = True
FORCE_QUANT = False
FORCE_FROZEN = False
FORCE_RETRAIN = False
FORCE_TAU = False
FORCE_FRONTIER = False
USE_TORCH_COMPILE = False  # Reliability first; flip True only after one clean run.

TARGET_TPS = 200.0
CALIBRATION_TOPK_LOGITS = 32  # Compact quality refs; enough for ranking, lighter on Drive.
TAU_DEPTH = 4

# Local facts are used only to prevent stale projection math from sneaking back
# in. Replace these after a clean local bench if the runtime changes.
LOCAL_QWEN3B_LOCKED_BASELINE_TPS = 30.0
LOCAL_QWEN3B_AWQ_OPTION_B_TPS = 23.8
LOCAL_QWEN3B_LAST_SPEC_GAIN = 0.0
QWEN_LOCKED_FAST_PATH_ENV = {
    'DISMANTLE_QWEN_TCB': '1',
    'DISMANTLE_QWEN_VOCAB_PRUNE': '32000',
    'DISMANTLE_QWEN_Q4K_LMHEAD': '1',
    'DISMANTLE_QWEN_FFN_DOWN_Q4K': '1',
    'DISMANTLE_QWEN_Q4K_PREDEC': '1',
}
LOCAL_ONLY_GATES = [
    'Convert/download the target GGUF on Mac and build a kernel profile.',
    'Run trace-dispatch with the trained head and inspect draft_accepted/draft_rejected.',
    'Run clean absolute benches with the locked fast-path env vars.',
    'Run W4A8 per-channel LM-head calibration locally if testing Track E.',
    'Keep AWQ Option B behind env gates until it beats predec locally.',
]

TARGETS = [
    {
        'name': 'qwen15b',
        'track': 'student_single_session',
        'enabled': RUN_STUDENT_SINGLE_SESSION,
        'model_id': 'Qwen/Qwen2.5-1.5B-Instruct',
        'artifact_dir': f'{DRIVE_ROOT}/qwen15b_artifacts',
        'corpus_dir': f'{DRIVE_ROOT}/qwen15b_corpus',
        'local_corpus': f'{LOCAL_ROOT}/qwen15b_corpus',
        'local_frozen': '/content/qwen15b_frozen.npz',
        'corpus_target_seqs': 16000,
        'calibration_batch': 16,
        'lm_head_chunk_tokens': 512,
        'train_max_rows': 16000,
        'train_max_row_tokens': 384,
        'train_epochs': 14,
        'train_batch': 48,
        'eval_max_windows': 20000,
        'frontier_max_depth': 16,
        'frontier_top_n': 4,
        # Offline ranking placeholder. Final TPS must come from local qwen1.5b GGUF benches.
        'base_tps': 85.0,
        'quant_multiplier': 1.00,
        'spec_efficiency': 0.80,
        'profile_hint': 'qwen15b-local-gguf-profile-required',
        'ship_default': False,
        'notes': 'Primary H100 target. Colab can rank heads; Mac runtime must validate net TPS.',
    },
    {
        'name': 'qwen3b',
        'track': 'qwen3b_reference_diagnostic',
        'enabled': RUN_QWEN3B_REFERENCE,
        'model_id': 'Qwen/Qwen2.5-3B-Instruct',
        'artifact_dir': f'{DRIVE_ROOT}/qwen3b_artifacts',
        'corpus_dir': f'{DRIVE_ROOT}/qwen3b_corpus',
        'local_corpus': f'{LOCAL_ROOT}/qwen3b_corpus',
        'local_frozen': '/content/qwen3b_frozen.npz',
        'corpus_target_seqs': 4000,
        'calibration_batch': 8,
        'lm_head_chunk_tokens': 384,
        'train_max_rows': 4000,
        'train_max_row_tokens': 192,
        'train_epochs': 6,
        'train_batch': 16,
        'eval_max_windows': 6000,
        'frontier_max_depth': 8,
        'frontier_top_n': 2,
        # No W4A8/spec optimism here: the latest local result says locked baseline wins.
        'base_tps': LOCAL_QWEN3B_LOCKED_BASELINE_TPS,
        'quant_multiplier': 1.00,
        'spec_efficiency': 1.00,
        'profile_hint': 'qwen3b-locked-predec-baseline',
        'ship_default': False,
        'notes': 'Optional offline acceptance diagnostic. Do not use as a 200 TPS aggregate proof.',
    },
]

TRAIN_SPECS = [
    {'family': 'b1_base', 'num_blocks': 1, 'head_heads': 16, 'head_ff_mult': 4.0, 'seq_len': 16, 'lr': 3e-4, 'residual_delta': 0.00, 'calib_weight': 0.10, 'seed': 0},
    {'family': 'b1_fast', 'num_blocks': 1, 'head_heads': 16, 'head_ff_mult': 4.0, 'seq_len': 16, 'lr': 6e-4, 'residual_delta': 0.00, 'calib_weight': 0.12, 'seed': 0},
    {'family': 'b1_hot',  'num_blocks': 1, 'head_heads': 16, 'head_ff_mult': 4.0, 'seq_len': 16, 'lr': 1e-3, 'residual_delta': 0.00, 'calib_weight': 0.12, 'seed': 0},
    {'family': 'b1_wide', 'num_blocks': 1, 'head_heads': 16, 'head_ff_mult': 6.0, 'seq_len': 16, 'lr': 6e-4, 'residual_delta': 0.00, 'calib_weight': 0.14, 'seed': 1},
    {'family': 'b2_rd_lo', 'num_blocks': 2, 'head_heads': 16, 'head_ff_mult': 4.0, 'seq_len': 16, 'lr': 3e-4, 'residual_delta': 0.01, 'calib_weight': 0.16, 'seed': 0},
    {'family': 'b2_rd_mid', 'num_blocks': 2, 'head_heads': 16, 'head_ff_mult': 4.0, 'seq_len': 16, 'lr': 6e-4, 'residual_delta': 0.02, 'calib_weight': 0.18, 'seed': 0},
    {'family': 'b2_rd_hot', 'num_blocks': 2, 'head_heads': 16, 'head_ff_mult': 4.0, 'seq_len': 16, 'lr': 1e-3, 'residual_delta': 0.02, 'calib_weight': 0.20, 'seed': 1},
]

for d in (DRIVE_ROOT, LOCAL_ROOT):
    os.makedirs(d, exist_ok=True)
for t in TARGETS:
    if not t['enabled']:
        continue
    for key in ('artifact_dir', 'corpus_dir', 'local_corpus'):
        os.makedirs(t[key], exist_ok=True)

if not os.path.exists(REPO_DIR):
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO, REPO_DIR])
else:
    run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH, '--depth', '1'])
    run(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'])
os.chdir(REPO_DIR)
GIT_SHA = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('repo:', GIT_SHA)

REQUIRED_FILES = [
    'colab/mega_calibrate.py',
    'colab/eagle5_train_pytorch.py',
    'colab/eagle5_tau_eval_pytorch.py',
    'colab/eagle5_frontier_policy.py',
    'colab/awq_w4a8_validate.py',
    'colab/build_qwen3b_frozen_hf.py',
    'colab/q2k_importance_calibrate.py',
    'tools/training/awq_calibrate.py',
    'tools/bench/quick_bench.sh',
    'tools/bench/eagle5_paired_bench.sh',
    'tools/bench/w4a8_lmhead_per_channel_bench.sh',
]
for path in REQUIRED_FILES:
    assert os.path.exists(path), f'missing required file: {path}'
    print('ok', path)

FEATURE_CHECKS = {
    'colab/eagle5_train_pytorch.py': ['--num-blocks', '--head-heads', '--head-ff-mult'],
    'colab/eagle5_tau_eval_pytorch.py': ['--num-blocks', '--head-heads', '--head-ff-mult'],
    'colab/eagle5_frontier_policy.py': ['--num-blocks', '--head-heads', '--head-ff-mult'],
}
missing_features = []
for path, flags in FEATURE_CHECKS.items():
    help_text = subprocess.check_output([sys.executable, path, '--help'], text=True, stderr=subprocess.STDOUT)
    for flag in flags:
        if flag not in help_text:
            missing_features.append(f'{path} lacks {flag}')
if missing_features:
    raise RuntimeError(
        'The cloned branch does not contain the required H100 pipeline changes. '
        'Push this workspace branch first or set BRANCH to a branch/commit that includes them. '
        + '\n'.join(f'- {m}' for m in missing_features)
    )

for path in ['colab/q2k_importance_calibrate.py', 'tools/training/awq_calibrate.py', 'colab/build_qwen3b_frozen_hf.py']:
    subprocess.check_output([sys.executable, path, '--help'], text=True, stderr=subprocess.STDOUT)
print('H100 artifact feature checks: ok')

print(json.dumps({
    'git_sha': GIT_SHA,
    'gpu': GPU_NAME,
    'target_tps': TARGET_TPS,
    'calibration_topk_logits': CALIBRATION_TOPK_LOGITS,
    'local_qwen3b_context': {
        'locked_baseline_tps': LOCAL_QWEN3B_LOCKED_BASELINE_TPS,
        'awq_option_b_tps': LOCAL_QWEN3B_AWQ_OPTION_B_TPS,
        'last_spec_gain': LOCAL_QWEN3B_LAST_SPEC_GAIN,
        'locked_env': QWEN_LOCKED_FAST_PATH_ENV,
    },
    'local_only_gates': LOCAL_ONLY_GATES,
    'train_specs': TRAIN_SPECS,
    'targets': [{k: v for k, v in t.items() if k != 'enabled' or v} for t in TARGETS],
}, indent=2))


In [ ]:
# Cell 2 - Helpers

def target_config(target):
    cfg = AutoConfig.from_pretrained(target['model_id'], trust_remote_code=False)
    n_layers = int(getattr(cfg, 'num_hidden_layers', 0) or getattr(cfg, 'n_layer', 0))
    hidden = int(getattr(cfg, 'hidden_size', 0) or getattr(cfg, 'n_embd', 0))
    vocab = int(getattr(cfg, 'vocab_size', 0))
    capture_layer = max(0, n_layers - 4)
    return {'n_layers': n_layers, 'hidden': hidden, 'vocab': vocab, 'capture_layer': capture_layer}


def stats_sequences(path):
    if not os.path.exists(path):
        return 0
    try:
        with np.load(path) as z:
            return int(np.asarray(z['sequences_written']).item()) if 'sequences_written' in z.files else 0
    except Exception as e:
        print(f'WARN: could not read stats file {path}: {e}')
        return 0


def corpus_state(target):
    stats_file = f"{target['corpus_dir']}/per_site_activation_stats.npz"
    shards = len(glob.glob(f"{target['corpus_dir']}/shard_*.parquet"))
    seqs = stats_sequences(stats_file)
    required_shards = math.ceil(target['corpus_target_seqs'] / 16)
    return {
        'shards': shards,
        'seqs': seqs,
        'required_shards': required_shards,
        'complete': shards >= required_shards and seqs >= target['corpus_target_seqs'],
    }


def run_calibration(target, info):
    state = corpus_state(target)
    print(f"[{target['name']}] corpus state: {state}")
    if state['complete']:
        return
    if not BUILD_CORPUS_IF_NEEDED:
        raise RuntimeError(f"{target['name']} corpus incomplete and BUILD_CORPUS_IF_NEEDED=False")

    run([
        sys.executable, 'colab/mega_calibrate.py',
        '--model', target['model_id'],
        '--max-sequences', str(target['corpus_target_seqs']),
        '--max-tokens', '2048',
        '--batch-size', str(target['calibration_batch']),
        '--capture-layer', str(info['capture_layer']),
        '--topk-logits', str(CALIBRATION_TOPK_LOGITS),
        '--lm-head-chunk-tokens', str(target['lm_head_chunk_tokens']),
        '--shard-size', '16',
        '--sync-dir', target['corpus_dir'],
        '--sync-every', '4',
        '--delete-local-after-sync',
        '--out', target['local_corpus'],
    ])
    print(f"[{target['name']}] corpus state after calibration: {corpus_state(target)}")


def build_quant_artifacts(target):
    stats_file = f"{target['corpus_dir']}/per_site_activation_stats.npz"
    assert os.path.exists(stats_file), f'missing activation stats: {stats_file}'
    out = {'stats': stats_file}
    if not RUN_QUANT_ARTIFACTS:
        return out

    awq_out = f"{target['artifact_dir']}/{target['name']}_awq_smoothing.json"
    q2_out = f"{target['artifact_dir']}/{target['name']}_q2k_importance.npz"
    if FORCE_QUANT or not os.path.exists(awq_out):
        run([
            sys.executable, 'tools/training/awq_calibrate.py',
            '--stats', stats_file,
            '--out', awq_out,
            '--alpha', '0.5',
            '--model', target['model_id'],
        ])
    else:
        print(f"[{target['name']}] skip AWQ artifact: {awq_out}")
    if FORCE_QUANT or not os.path.exists(q2_out):
        run([
            sys.executable, 'colab/q2k_importance_calibrate.py',
            '--stats', stats_file,
            '--out', q2_out,
            '--model', target['model_id'],
            '--mean-weight', '0.70',
            '--max-weight', '0.30',
        ])
    else:
        print(f"[{target['name']}] skip Q2/IQ2 importance artifact: {q2_out}")
    out.update({'awq': awq_out, 'q2_importance': q2_out})
    return out


def build_frozen(target):
    frozen_out = f"{target['artifact_dir']}/{target['name']}_frozen.npz"
    if FORCE_FROZEN or not os.path.exists(frozen_out):
        run([
            sys.executable, 'colab/build_qwen3b_frozen_hf.py',
            '--model', target['model_id'],
            '--out', frozen_out,
        ])
    else:
        print(f"[{target['name']}] skip frozen baseline: {frozen_out}")
    assert os.path.exists(frozen_out), f'missing frozen baseline: {frozen_out}'
    return frozen_out


def lr_tag(lr):
    return f'{lr:.0e}'.replace('+', '').replace('-0', '-')


def ff_tag(ff_mult):
    return f'{int(round(float(ff_mult) * 10)):02d}'


def spec_tag(spec):
    rd = int(round(spec['residual_delta'] * 1000))
    cw = int(round(spec['calib_weight'] * 100))
    return (
        f"{spec['family']}_b{spec['num_blocks']}_h{spec['head_heads']}_"
        f"ff{ff_tag(spec['head_ff_mult'])}_s{spec['seq_len']}_"
        f"lr{lr_tag(spec['lr'])}_rd{rd:03d}_cw{cw:02d}_seed{spec['seed']}"
    )


def stage_frozen(target, frozen_out):
    if not os.path.exists(target['local_frozen']) or os.path.getsize(target['local_frozen']) != os.path.getsize(frozen_out):
        print(f"[{target['name']}] stage frozen -> {target['local_frozen']}")
        shutil.copyfile(frozen_out, target['local_frozen'])
    return target['local_frozen']


def env_line(env):
    return ' '.join(f'{k}={v}' for k, v in env.items())


In [ ]:
# Cell 3 - Build corpora, quant artifacts, and frozen baselines
TARGET_INFO = {}
TARGET_ARTIFACTS = {}
for target in TARGETS:
    if not target['enabled']:
        print(f"skip disabled target: {target['name']} ({target['track']})")
        continue
    info = target_config(target)
    TARGET_INFO[target['name']] = info
    print(f"\n=== {target['name']} {target['model_id']} cfg={info}")
    run_calibration(target, info)
    quant = build_quant_artifacts(target)
    frozen = build_frozen(target)
    TARGET_ARTIFACTS[target['name']] = {**quant, 'frozen': frozen}
    print(json.dumps(TARGET_ARTIFACTS[target['name']], indent=2))


In [ ]:
# Cell 4 - Train Eagle5 grids and run tau/frontier ranking
ALL_RESULTS = {}
if not RUN_EAGLE5_GRID:
    print('RUN_EAGLE5_GRID=False; skipping head training and policy ranking.')
else:
    for target in TARGETS:
        if not target['enabled']:
            continue
        name = target['name']
        info = TARGET_INFO[name]
        artifacts = TARGET_ARTIFACTS[name]
        local_frozen = stage_frozen(target, artifacts['frozen'])
        gc.collect()
        torch.cuda.empty_cache()

        trained = []
        for spec in TRAIN_SPECS:
            tag = spec_tag(spec)
            ckpt_dir = f"{target['artifact_dir']}/checkpoints/eagle5_{name}_{tag}"
            final_head = f'{ckpt_dir}/head_final.safetensors'
            os.makedirs(ckpt_dir, exist_ok=True)
            if os.path.exists(final_head) and not FORCE_RETRAIN:
                print(f"[{name}] skip trained {tag}: {final_head}")
            else:
                print(f"\n=== [{name}] train {tag} spec={spec}")
                gc.collect()
                torch.cuda.empty_cache()
                cmd = [
                    sys.executable, 'colab/eagle5_train_pytorch.py',
                    '--corpus-dir', target['corpus_dir'],
                    '--frozen', local_frozen,
                    '--ckpt-dir', ckpt_dir,
                    '--epochs', str(target['train_epochs']),
                    '--batch-size', str(target['train_batch']),
                    '--seq-len', str(spec['seq_len']),
                    '--lr', str(spec['lr']),
                    '--num-blocks', str(spec['num_blocks']),
                    '--head-heads', str(spec['head_heads']),
                    '--head-ff-mult', str(spec['head_ff_mult']),
                    '--capture-layer', str(info['capture_layer']),
                    '--max-rows', str(target['train_max_rows']),
                    '--max-row-tokens', str(target['train_max_row_tokens']),
                    '--sparsity-head', 'off',
                    '--seed', str(spec['seed']),
                    '--calib-loss-weight', str(spec['calib_weight']),
                    '--residual-delta-loss-weight', str(spec['residual_delta']),
                    '--save-safetensors',
                ]
                if USE_TORCH_COMPILE:
                    cmd.append('--compile')
                run(cmd)
            assert os.path.exists(final_head), f'training did not write {final_head}'
            trained.append((tag, spec, final_head))

        tau_results = {}
        for tag, spec, ckpt in trained:
            out_path = f"{target['artifact_dir']}/eagle5_tau_{name}_{tag}.json"
            if os.path.exists(out_path) and not FORCE_TAU:
                print(f"[{name}] skip tau {tag}: {out_path}")
            else:
                print(f"\n=== [{name}] tau eval {tag}")
                gc.collect()
                torch.cuda.empty_cache()
                run([
                    sys.executable, 'colab/eagle5_tau_eval_pytorch.py',
                    '--ckpt', ckpt,
                    '--frozen', local_frozen,
                    '--corpus', target['corpus_dir'],
                    '--out', out_path,
                    '--depth', str(TAU_DEPTH),
                    '--max-windows', str(target['eval_max_windows']),
                    '--max-row-tokens', str(target['train_max_row_tokens']),
                    '--num-blocks', str(spec['num_blocks']),
                    '--head-heads', str(spec['head_heads']),
                    '--head-ff-mult', str(spec['head_ff_mult']),
                    '--base-tps', str(target['base_tps']),
                    '--w4a8-multiplier', str(target['quant_multiplier']),
                    '--spec-efficiency', str(target['spec_efficiency']),
                ])
            with open(out_path) as f:
                tau_results[tag] = json.load(f)
            tau_results[tag]['train_spec'] = spec
            tau_results[tag]['ckpt'] = ckpt

        ranked_tau = sorted(
            tau_results.items(),
            key=lambda kv: (kv[1]['projection']['projected_dec_tps'], kv[1]['tau'], kv[1]['depth1_accept_rate']),
            reverse=True,
        )
        tau_winner_path = f"{target['artifact_dir']}/eagle5_tau_winner_{name}.json"
        with open(tau_winner_path, 'w') as f:
            json.dump({'winner': ranked_tau[0][0], 'result': ranked_tau[0][1], 'all': tau_results}, f, indent=2, sort_keys=True)

        print(f"\n[{name}] Tau race results:")
        for tag, r in ranked_tau:
            print(f"  {tag:58s} tau={r['tau']:.3f} depth1={r['depth1_accept_rate']:.2%} offline_projected={r['projection']['projected_dec_tps']:.1f}")

        frontier_results = {}
        for tag, r in ranked_tau[:target['frontier_top_n']]:
            spec = r['train_spec']
            out_path = f"{target['artifact_dir']}/eagle5_frontier_{name}_{tag}.json"
            if os.path.exists(out_path) and not FORCE_FRONTIER:
                print(f"[{name}] skip frontier {tag}: {out_path}")
            else:
                print(f"\n=== [{name}] frontier search {tag}")
                run([
                    sys.executable, 'colab/eagle5_frontier_policy.py',
                    '--ckpt', r['ckpt'],
                    '--frozen', local_frozen,
                    '--corpus', target['corpus_dir'],
                    '--out', out_path,
                    '--max-depth', str(target['frontier_max_depth']),
                    '--depths', '2,4,6,8,12,16',
                    '--lattice-widths', '2,3,4,6',
                    '--max-windows', str(target['eval_max_windows']),
                    '--max-row-tokens', str(target['train_max_row_tokens']),
                    '--eval-batch-size', '192',
                    '--num-blocks', str(spec['num_blocks']),
                    '--head-heads', str(spec['head_heads']),
                    '--head-ff-mult', str(spec['head_ff_mult']),
                    '--base-tps', str(target['base_tps']),
                    '--w4a8-multiplier', str(target['quant_multiplier']),
                    '--spec-efficiency', str(target['spec_efficiency']),
                ])
            with open(out_path) as f:
                frontier_results[tag] = json.load(f)

        frontier_ranked = sorted(
            frontier_results.items(),
            key=lambda kv: kv[1]['policies']['best_deployable']['projected_dec_tps'],
            reverse=True,
        )
        frontier_winner_path = f"{target['artifact_dir']}/eagle5_frontier_winner_{name}.json"
        with open(frontier_winner_path, 'w') as f:
            json.dump({'winner': frontier_ranked[0][0], 'result': frontier_ranked[0][1], 'all': frontier_results}, f, indent=2, sort_keys=True)

        print(f"\n[{name}] Frontier results:")
        for tag, payload in frontier_ranked:
            best = payload['policies']['best_deployable']
            overall = payload['policies']['best_overall']
            print(f"  {tag:58s} deployable={best['kind']:34s} offline_projected={best['projected_dec_tps']:.1f} overall={overall['projected_dec_tps']:.1f}")

        ALL_RESULTS[name] = {
            'target': target,
            'info': info,
            'artifacts': artifacts,
            'tau_winner_path': tau_winner_path,
            'frontier_winner_path': frontier_winner_path,
            'ranked_tau': ranked_tau,
            'frontier_ranked': frontier_ranked,
        }


In [ ]:
# Cell 5 - Handoff manifest and local-only commands
summary_path = f'{DRIVE_ROOT}/past200_summary.md'
json_summary_path = f'{DRIVE_ROOT}/past200_summary.json'
locked_env = env_line(QWEN_LOCKED_FAST_PATH_ENV)
summary = {
    'schema': 'dismantle-past200-h100-artifact-v2',
    'git_sha': GIT_SHA,
    'gpu': GPU_NAME,
    'target_tps': TARGET_TPS,
    'calibration_topk_logits': CALIBRATION_TOPK_LOGITS,
    'local_qwen3b_context': {
        'locked_baseline_tps': LOCAL_QWEN3B_LOCKED_BASELINE_TPS,
        'awq_option_b_tps': LOCAL_QWEN3B_AWQ_OPTION_B_TPS,
        'last_spec_gain': LOCAL_QWEN3B_LAST_SPEC_GAIN,
        'locked_env': QWEN_LOCKED_FAST_PATH_ENV,
    },
    'local_only_gates': LOCAL_ONLY_GATES,
    'targets': {},
}

lines = [
    '# Past-200 H100 Artifact Handoff',
    '',
    f'GPU: {GPU_NAME}',
    f'Git SHA: {GIT_SHA}',
    f'Target TPS: {TARGET_TPS:.1f}',
    '',
    '## Local Context',
    '',
    f'- Qwen-3B locked baseline remains the shipping path until displaced locally: about {LOCAL_QWEN3B_LOCKED_BASELINE_TPS:.1f} dec_tps.',
    f'- AWQ Option B plus W4A8 is held: last local result was about {LOCAL_QWEN3B_AWQ_OPTION_B_TPS:.1f} dec_tps after disabling predec.',
    f'- Eagle5 needs trace-dispatch proof before promotion: last observed local gain was {LOCAL_QWEN3B_LAST_SPEC_GAIN:.1f}.',
    '- Colab projections below are offline ranking signals, not ship decisions.',
    '',
    '## Results',
    '',
    '| Target | Track | Winner | Policy | Accepted Drafts/Verify | Offline TPS | Gate |',
    '|---|---|---|---|---:|---:|---|',
]

for name, result in ALL_RESULTS.items():
    target = result['target']
    frontier_info = json.load(open(result['frontier_winner_path']))
    tau_info = json.load(open(result['tau_winner_path']))
    winner_tag = frontier_info['winner']
    policy = frontier_info['result']['policies']['best_deployable']
    hints = frontier_info['result']['runtime_hints']
    offline_tps = float(policy['projected_dec_tps'])
    accepted = float(policy.get('accepted_draft_tokens_per_verify', 0.0))

    if name == 'qwen15b':
        gate = 'LOCAL_BENCH_REQUIRED' if offline_tps >= TARGET_TPS else 'OFFLINE_SHORT'
        projection_scope = 'Student offline rank only; replace base_tps after local qwen1.5b GGUF bench.'
    else:
        gate = 'DIAGNOSTIC_ONLY'
        projection_scope = 'Qwen-3B diagnostic only; locked predec baseline remains ship config.'

    lines.append(f'| {name} | {target["track"]} | {winner_tag} | {policy["kind"]} | {accepted:.2f} | {offline_tps:.1f} | {gate} |')

    summary['targets'][name] = {
        'track': target['track'],
        'model_id': target['model_id'],
        'profile_hint': target['profile_hint'],
        'ship_default': target['ship_default'],
        'notes': target['notes'],
        'winner': winner_tag,
        'tau_winner': tau_info['winner'],
        'best_policy': policy,
        'runtime_hints': hints,
        'offline_projected_tps': offline_tps,
        'projection_scope': projection_scope,
        'gate': gate,
        'artifacts': result['artifacts'],
        'frontier_winner_path': result['frontier_winner_path'],
        'tau_winner_path': result['tau_winner_path'],
    }

lines.extend(['', '## Runtime Hints', ''])
for name, payload in summary['targets'].items():
    lines.extend([
        f'### {name}',
        '',
        '~~~json',
        json.dumps(payload['runtime_hints'], indent=2, sort_keys=True),
        '~~~',
        '',
        f'Projection scope: {payload["projection_scope"]}',
        '',
    ])

lines.extend(['## Artifacts', ''])
for name, payload in summary['targets'].items():
    lines.append(f'### {name}')
    for key, path in payload['artifacts'].items():
        size = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0.0
        lines.append(f'- {key}: {path} ({size:.1f} MB)')
    lines.append(f'- tau: {payload["tau_winner_path"]}')
    lines.append(f'- frontier: {payload["frontier_winner_path"]}')
    lines.append('')

lines.extend(['## Local-Only Gates', ''])
for gate in LOCAL_ONLY_GATES:
    lines.append(f'- {gate}')
lines.append('')

student = summary['targets'].get('qwen15b')
qwen3b = summary['targets'].get('qwen3b')
lines.extend(['## Local Commands', ''])
lines.extend([
    '### Shipping Qwen-3B baseline sanity check',
    '',
    '~~~bash',
    f'{locked_env} WEIGHTS=models/qwen2.5-3b-instruct-q4_k_m.gguf PROFILE=profiles/qwen3b-instruct-q4k.m3pro18.json TOKENS=64 bash tools/bench/quick_bench.sh --strict',
    '~~~',
    '',
])

if student:
    student_head = f'artifacts/qwen15b_artifacts/checkpoints/eagle5_qwen15b_{student["winner"]}/head_final.safetensors'
    lines.extend([
        '### Student spec-decode trace',
        '',
        'Requires local qwen1.5b GGUF plus a kernel profile. This is the first real gate for the H100 head.',
        '',
        '~~~bash',
        f'{locked_env} ./target/release/dismantle generate --trace-dispatch --weights models/qwen2.5-1.5b-instruct-q4_k_m.gguf --prompt "Once upon a time" --max-new-tokens 64 --temperature 0.0 --kernel-profile profiles/qwen15b-instruct-q4k.m3pro18.json --speculate eagle5 --verify-window 4 --eagle5-head {student_head}',
        '~~~',
        '',
        '### Student paired bench after trace accepts drafts',
        '',
        '~~~bash',
        f'{locked_env} WEIGHTS=models/qwen2.5-1.5b-instruct-q4_k_m.gguf PROFILE=profiles/qwen15b-instruct-q4k.m3pro18.json EAGLE5_HEAD={student_head} TRIALS=10 TOKENS=128 bash tools/bench/eagle5_paired_bench.sh',
        '~~~',
        '',
    ])

if qwen3b:
    qwen3b_head = f'artifacts/qwen3b_artifacts/checkpoints/eagle5_qwen3b_{qwen3b["winner"]}/head_final.safetensors'
    lines.extend([
        '### Optional Qwen-3B spec diagnostic',
        '',
        'Do this only to debug acceptance/runtime wiring. It is not an aggregate 200 TPS proof.',
        '',
        '~~~bash',
        f'{locked_env} WEIGHTS=models/qwen2.5-3b-instruct-q4_k_m.gguf PROFILE=profiles/qwen3b-instruct-q4k.m3pro18.json EAGLE5_HEAD={qwen3b_head} TRIALS=5 TOKENS=64 bash tools/bench/eagle5_paired_bench.sh',
        '~~~',
        '',
    ])

lines.extend([
    '### Held W4A8/AWQ experiments',
    '',
    'Run these only as experiments. AWQ Option B currently disables predec and lost throughput locally.',
    '',
    '~~~bash',
    'cargo test --release -p dismantle-core --test w4a8_per_channel_calibrate -- --ignored --nocapture --test-threads=1',
    'TOKENS=64 TRIALS=5 WEIGHTS=models/qwen2.5-3b-instruct-q4_k_m.gguf PROFILE=profiles/qwen3b-instruct-q4k.m3pro18.json bash tools/bench/w4a8_lmhead_per_channel_bench.sh',
    '~~~',
    '',
    '~~~bash',
    'DISMANTLE_QWEN_AWQ=1 DISMANTLE_QWEN_W4A8=1 DISMANTLE_QWEN_Q4K_PREDEC=0 DISMANTLE_QWEN_AWQ_SMOOTHING=profiles/qwen3b_awq_smoothing.json WEIGHTS=models/qwen2.5-3b-instruct-q4_k_m.gguf PROFILE=profiles/qwen3b-instruct-q4k.m3pro18.json TRIALS=5 TOKENS=64 bash tools/bench/eagle5_paired_bench.sh',
    '~~~',
    '',
])

with open(summary_path, 'w') as f:
    f.write('\n'.join(lines) + '\n')
with open(json_summary_path, 'w') as f:
    json.dump(summary, f, indent=2, sort_keys=True)

print('\n'.join(lines))
print(f'\nhandoff: {summary_path}')
print(f'json: {json_summary_path}')
